### Install New Libraries

In [ ]:
!pip install ddgs trafilatura -q

### Setup

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv
import json
from pprint import pprint
from ddgs import DDGS
import trafilatura

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing.")

client = OpenAI(api_key=OPENAI_API_KEY)

### Step 1: Define the Tools

In [ ]:
def search_web(query: str):
    # Search the web using DuckDuckGo browser. Returns 3 results.
    ddgs = DDGS()
    results = ddgs.text(query, max_results=3)
    print(f"  \u2705 Got Results\n")
    return json.dumps(results, indent=2)

In [ ]:
def fetch_url(url: str):
    # Fetch the URL content using Trafilatura. Returns the extracted text if successful, otherwise None.
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        text = trafilatura.extract(downloaded)
        if text:
            print(f"  \u2705 Got text: {len(text)} characters\n")
            return text
    print(f"  \u274C Failed to fetch or extracte text from {url}\n")
    return f"Could not fetch or extract text from {url}. Try a different source."

In [ ]:
# Test the search_web function
search_web("AI in healthcare in 2030")

In [ ]:
# Test the fetch_url function
result = fetch_url("https://en.wikipedia.org/wiki/Artificial_intelligence_in_healthcare")
print(result)

### Step 2: Describe as LLM tools

In [ ]:
tools = []

In [ ]:
search_web_function = {
    "name": "search_web",
    "description": "Search the web using DuckDuckGo browser. Returns 3 results.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query"
            }
        },
        "required": ["query"]
    }
}

# Add the search_web function to the list of available tools
tools.append({"type": "function", "function": search_web_function})

In [ ]:
fetch_url_function = {
    "name": "fetch_url",
    "description": "Fetch the content of a URL using trafilatura.",
    "parameters": {
        "type": "object",
        "properties": {
            "url": {
                "type": "string",
                "description": "The URL to fetch"
            }
        },
        "required": ["url"]
    }
}

# Add to the list of available tools
tools.append({"type": "function", "function": fetch_url_function})

In [ ]:
# Check the list of available tools
for tool in tools:
    print(tool["function"]["name"])

### Step 3: Tool Call Handler

In [ ]:
def handle_tool_call(tool_calls):
    tool_results = []

    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)

        print(f"  \U0001f527 Handling tool call for function: {function_name} with arguments: {args}") # For debugging

        # Route to the appropriate function based on function_name
        if function_name == "search_web":
            # Actually perform the web search, i.e. call the tool
            result = search_web(args["query"])
            content = f"Search results: {result}"
        elif function_name == "fetch_url":
            result = fetch_url(args["url"])
            content = f"Fetched URL content: {result}"
        else:
            content = f"Unknown tool call: {function_name}"

        tool_results.append({
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id
        })

    # Return what to add to the context about tool call results, a list of dictionaries
    return tool_results


### Step 4: The System Prompt

This tells the LLM who it is and how to behave. The key things:

- What its job is
- What tools it has
- What process to follow
- What output format to produce

In [ ]:
RESEARCH_AGENT_PROMPT = """ You are a research agent. 

Your job is to assist users by gathering comprehensive information from the web and fetching detailed content from URLs as requested.

You have access to the following tools:
- search_web: Search the web using DuckDuckGo to find relevant results for a query
- fetch_url: Fetch and extract the full text content from a URL

Your process should be:
1. Understand the user's research request
2. Use search_web to find relevant sources and links
3. Use fetch_url to retrieve detailed content from promising URLs
4. Synthesize the gathered information into a coherent response

Your output format should be:
- Provide clear, concise, and relevant information
- Cite sources and include relevant URLs
- Organize findings in a logical, easy-to-read manner
- Highlight key insights and important details from the fetched content
"""

### Step 5: The Agentic Loop